# Stage 7 Capsule / Weld-Feature Planning Geometry

This notebook compares simulated optical/proxy geometry with a
capsule-like application target. These capsule/weld-feature results
are application-planning geometry proxies. They do not predict actual
weld success, bonding, void formation, ablation, or refractive-index
change without experimental calibration.

A pass score means the simulated optical/proxy geometry matches the
requested target under the model assumptions. It does not mean the
material will physically produce that feature.


In [1]:
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import bessel_twin_core as bt
from vbb_study import setup_study, vbb_capsule
from vbb_study.publication import capsule as capsule_schema

PATHS = setup_study.bootstrap(Path.cwd())
CSV_OUT = PATHS["csv"] / "capsule"
CSV_COMPAT = PATHS["csv"] / "stage9_capsule"
FIG_OUT = PATHS["figures"] / "capsule"
CSV_OUT.mkdir(parents=True, exist_ok=True)
CSV_COMPAT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 90)


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [2]:
# STAGE88: visible editable controls for exploratory notebook use.
# Edit NOTEBOOK_CONTROLS below and re-run this cell to apply parameter
# overrides for downstream capsule design cells.
from vbb_study.publication import notebook_controls as nb_controls

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(
    stage='capsule',
    # ── edit these for capsule geometry ─────────────────────────────────────
    target_width_um=4.2,
    target_length_um=150.0,
)

# Wire control parameters to named variables used by downstream cells.
_p = NOTEBOOK_CONTROLS.parameters or {}
TARGET_WIDTH_um = float(_p.get("target_width_um", 4.2))
TARGET_LENGTH_um = float(_p.get("target_length_um", 150.0))

try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


,control,value
0,stage,capsule
1,run_mode,balanced
2,save_outputs,False
3,use_canonical_outputs,True
4,allow_publication_export,False
5,notes,Edit for exploration; keep QA labels/caveats v...
6,target_width_um,4.2
7,target_length_um,150.0


In [3]:
# Interactive beam quicklook — adjust sliders and click "Update plots".
# Runs a fast preview only; nothing is saved and this is independent of the
# capsule design cells below.
from vbb_study.publication import notebook_widgets as nbw

_ql_base = bt.default_config("fast")
_panel = nbw.interactive_quicklook(_ql_base, method='holographic', preset='fast')
display(_panel)


## Target And QA Boundary

The target is a capsule-like XZ outline with a transverse width near
4.2 um and length/depth near 200 um. The comparison uses optical
geometry, thresholded fluence proxies, and propagation QA labels.
`fail` propagation rows are not design-ready. `marginal` rows are
exploratory. `pass` rows are still planning proxies unless
experimentally calibrated.


In [4]:
TARGET_WIDTH_UM = 4.2
TARGET_LENGTH_UM = 200.0
TARGET_DEPTH_UM = 200.0

base = bt.default_config("fast")
capsule_config = replace(
    base,
    grid=replace(
        base.grid,
        N=256,
        ideal_N=160,
        ideal_dx_m=0.30 * bt.um,
        crop_pixels=112,
        axial_points=9,
        axial_range_m=120.0 * bt.um,
        axial_target_factor=1.4,
        device_downsample=8,
        label="fast",
    ),
    target=replace(
        base.target,
        ell=0,
        target_core_diameter_m=TARGET_WIDTH_UM * bt.um,
        target_bessel_length_m=TARGET_LENGTH_UM * bt.um,
    ),
    energy=replace(base.energy, pulse_energy_in_J=20.0 * bt.uJ),
)

print(f"Target width={TARGET_WIDTH_UM:.1f} um, length={TARGET_LENGTH_UM:.0f} um")


Target width=4.2 um, length=200 um


## Optical Candidate Ranking

The inverse-design sweep is optical-only. It ranks candidate spot/zone
geometry and hardware reachability before any thresholded material
proxy is used.


In [5]:
design_df = vbb_capsule.design_solver(
    capsule_config,
    spot_range_um=(2.0, 5.0),
    target_zone_um=TARGET_LENGTH_UM,
    wavelength_m=capsule_config.laser.wavelength_m,
    ell_values=(0, 1, 2, 3),
    spot_samples_um=(2.0, 3.0, TARGET_WIDTH_UM, 5.0),
    energy_samples_uJ=(10.0, 20.0, 40.0),
)

ranking = vbb_capsule.candidate_ranking_from_design_solver(
    design_df,
    target_width_um=TARGET_WIDTH_UM,
    target_length_um=TARGET_LENGTH_UM,
    target_depth_um=TARGET_DEPTH_UM,
)
ranking_path = CSV_OUT / "capsule_candidate_ranking.csv"
ranking.to_csv(ranking_path, index=False)
ranking.to_csv(CSV_COMPAT / "09_capsule_design_feasible_set.csv", index=False)

display_cols = [
    "planning_rank",
    "case_id",
    "geometry_model_status",
    "material_model_status",
    "calibration_status",
    "predicted_width_um",
    "predicted_length_um",
    "capsule_fit_score",
    "capsule_acceptance_label",
    "hardware_status",
]
display(ranking[display_cols].head(12))
print(ranking_path)


,planning_rank,case_id,geometry_model_status,material_model_status,calibration_status,predicted_width_um,predicted_length_um,capsule_fit_score,capsule_acceptance_label,hardware_status
0,1,capsule_candidate_ell0_w4.2_e10,optical_geometry_proxy,optical_only,uncalibrated,4.200000,200.0,1.000000,exploratory_proxy,lab_proxy
1,2,capsule_candidate_ell0_w4.2_e20,optical_geometry_proxy,optical_only,uncalibrated,4.200000,200.0,1.000000,exploratory_proxy,lab_proxy
2,3,capsule_candidate_ell0_w4.2_e40,optical_geometry_proxy,optical_only,uncalibrated,4.200000,200.0,1.000000,exploratory_proxy,lab_proxy
3,4,capsule_candidate_ell1_w5.0_e10,optical_geometry_proxy,optical_only,uncalibrated,3.827825,200.0,0.959300,exploratory_proxy,lab_proxy
4,5,capsule_candidate_ell1_w5.0_e20,optical_geometry_proxy,optical_only,uncalibrated,3.827825,200.0,0.959300,exploratory_proxy,lab_proxy
5,6,capsule_candidate_ell1_w5.0_e40,optical_geometry_proxy,optical_only,uncalibrated,3.827825,200.0,0.959300,exploratory_proxy,lab_proxy
6,7,capsule_candidate_ell2_w3.0_e10,optical_geometry_proxy,optical_only,uncalibrated,3.809859,200.0,0.957502,exploratory_proxy,lab_proxy
7,8,capsule_candidate_ell2_w3.0_e20,optical_geometry_proxy,optical_only,uncalibrated,3.809859,200.0,0.957502,exploratory_proxy,lab_proxy
8,9,capsule_candidate_ell2_w3.0_e40,optical_geometry_proxy,optical_only,uncalibrated,3.809859,200.0,0.957502,exploratory_proxy,lab_proxy
9,10,capsule_candidate_ell3_w2.0_e10,optical_geometry_proxy,optical_only,uncalibrated,3.493712,200.0,0.928022,exploratory_proxy,lab_proxy


C:\PhD\Code\Publication_Study\outputs\csv\capsule\capsule_candidate_ranking.csv


## Thresholded Capsule Geometry Proxy

The sweep compares ideal and lab-realistic scalar propagation paths.
The XZ map is a normalised planning visualisation that preserves the
axial envelope for geometry comparison. It is not an energy-conserving
3D deposition model and not a weld/material-response predictor.


In [6]:
z_values = np.linspace(0.0, 250.0 * bt.um, 9)
strengths = (0.0, 0.35, 0.70, 1.00)
sweep_df, capsule_cases = vbb_capsule.sweep_capsule_apodization(
    capsule_config,
    strengths=strengths,
    z_values_m=z_values,
)

summary = vbb_capsule.capsule_summary_from_cases(
    capsule_cases,
    target_width_um=TARGET_WIDTH_UM,
    target_length_um=TARGET_LENGTH_UM,
    target_depth_um=TARGET_DEPTH_UM,
)
summary_path = CSV_OUT / "capsule_weld_feature_design_summary.csv"
summary.to_csv(summary_path, index=False)
summary.to_csv(CSV_COMPAT / "09_capsule_apodization_sweep.csv", index=False)

display_cols = [
    "case_id",
    "path",
    "geometry_model_status",
    "material_model_status",
    "calibration_status",
    "propagation_power_label",
    "predicted_width_um",
    "predicted_length_um",
    "overlap_score",
    "capsule_fit_score",
    "capsule_acceptance_label",
    "xz_energy_conservation_status",
]
display(summary[display_cols])
print(summary_path)


,case_id,path,geometry_model_status,material_model_status,calibration_status,propagation_power_label,predicted_width_um,predicted_length_um,overlap_score,capsule_fit_score,capsule_acceptance_label,xz_energy_conservation_status
0,capsule_ideal_a0.00,ideal,thresholded_fluence_proxy,planning_proxy,uncalibrated,pass,15.633333,250.0,0.193878,0.463562,exploratory_proxy,normalised_visualisation
1,capsule_realistic_a0.00,realistic,thresholded_fluence_proxy,planning_proxy,uncalibrated,pass,13.928792,250.0,0.283019,0.487344,exploratory_proxy,normalised_visualisation
2,capsule_ideal_a0.35,ideal,thresholded_fluence_proxy,planning_proxy,uncalibrated,pass,18.433333,250.0,0.147287,0.433083,exploratory_proxy,normalised_visualisation
3,capsule_realistic_a0.35,realistic,thresholded_fluence_proxy,planning_proxy,uncalibrated,pass,19.120687,250.0,0.235872,0.505232,exploratory_proxy,normalised_visualisation
4,capsule_ideal_a0.70,ideal,thresholded_fluence_proxy,planning_proxy,uncalibrated,pass,20.633333,250.0,0.130137,0.442947,exploratory_proxy,normalised_visualisation
5,capsule_realistic_a0.70,realistic,thresholded_fluence_proxy,planning_proxy,uncalibrated,pass,22.861084,250.0,0.195918,0.535964,exploratory_proxy,normalised_visualisation
6,capsule_ideal_a1.00,ideal,thresholded_fluence_proxy,planning_proxy,uncalibrated,pass,16.366667,250.0,0.162996,0.447378,exploratory_proxy,normalised_visualisation
7,capsule_realistic_a1.00,realistic,thresholded_fluence_proxy,planning_proxy,uncalibrated,pass,20.962972,250.0,0.216216,0.552921,planning_proxy_candidate,normalised_visualisation


C:\PhD\Code\Publication_Study\outputs\csv\capsule\capsule_weld_feature_design_summary.csv


## Acceptance Summary

Acceptance is a planning label. It means the simulated proxy geometry
is useful for comparing designs under the current assumptions. It does
not mean that a physical weld, bond, void, ablation mark, or refractive
index change will occur.


In [7]:
acceptance = vbb_capsule.capsule_acceptance_summary(summary)
acceptance_path = CSV_OUT / "capsule_acceptance_summary.csv"
acceptance.to_csv(acceptance_path, index=False)
acceptance.to_csv(CSV_COMPAT / "09_capsule_acceptance_summary.csv", index=False)

display_cols = [
    "case_id",
    "acceptance_pass",
    "capsule_acceptance_label",
    "propagation_power_label",
    "geometry_model_status",
    "material_model_status",
    "calibration_status",
    "actual_weld_success_claimed",
]
display(acceptance[display_cols])
assert not acceptance["actual_weld_success_claimed"].astype(bool).any()
assert not (
    (acceptance["calibration_status"] == "uncalibrated")
    & (acceptance["material_model_status"] == "experimentally_calibrated")
).any()
print(acceptance_path)


,case_id,acceptance_pass,capsule_acceptance_label,propagation_power_label,geometry_model_status,material_model_status,calibration_status,actual_weld_success_claimed
0,capsule_ideal_a0.00,False,exploratory_proxy,pass,thresholded_fluence_proxy,planning_proxy,uncalibrated,False
1,capsule_realistic_a0.00,False,exploratory_proxy,pass,thresholded_fluence_proxy,planning_proxy,uncalibrated,False
2,capsule_ideal_a0.35,False,exploratory_proxy,pass,thresholded_fluence_proxy,planning_proxy,uncalibrated,False
3,capsule_realistic_a0.35,False,exploratory_proxy,pass,thresholded_fluence_proxy,planning_proxy,uncalibrated,False
4,capsule_ideal_a0.70,False,exploratory_proxy,pass,thresholded_fluence_proxy,planning_proxy,uncalibrated,False
5,capsule_realistic_a0.70,False,exploratory_proxy,pass,thresholded_fluence_proxy,planning_proxy,uncalibrated,False
6,capsule_ideal_a1.00,False,exploratory_proxy,pass,thresholded_fluence_proxy,planning_proxy,uncalibrated,False
7,capsule_realistic_a1.00,True,planning_proxy_candidate,pass,thresholded_fluence_proxy,planning_proxy,uncalibrated,False


C:\PhD\Code\Publication_Study\outputs\csv\capsule\capsule_acceptance_summary.csv
